# N002 · 数值、布尔值与运算符

**目标：** 根据题意选择类型并写出数值表达式。

**先修：** N001。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** int/float/bool/None；整除与取模；比较运算；浮点近似。

**配套讲解来源：** [同名逐章意见](../../comment/002_values_types_operators.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决两道很基础但很典型的题，背后练的是"根据题意选对类型、写对表达式"。

**问题 1（最小偶数倍）**：给你一个正整数 n，让你找"最小的、同时是 n 的倍数、又是偶数"的那个正整数。具体例子：输入 `n=5`，输出 `10`——因为 5 的倍数依次是 5、10、15……其中 5 是奇数不合格，10 既是 5 的倍数又是偶数，所以答案就是 10；而输入 `n=6` 时，输出就是 `6` 本身，因为 6 已经既是 6 的倍数又是偶数，没有比它更小的了。注意答案只有两种形态：要么是 n、要么是 2n，这个"二选一"结构是解题的钥匙。

**问题 2（温度转换）**：给你一个摄氏温度，要同时转换成开氏温度和华氏温度，用列表 `[开氏, 华氏]` 返回。具体例子：输入 `celsius=0`，输出 `[273.15, 32.0]`——因为开氏 = 摄氏 + 273.15（0 + 273.15 = 273.15），华氏 = 摄氏 × 1.8 + 32（0 × 1.8 + 32 = 32）。

两道题虽然简单，却分别代表两类常见题型：问题 1 是"分析条件后一步出答案"的推理题（没有循环、没有扫描）；问题 2 是"公式直译"的计算题（把数学关系忠实抄成表达式）。

这一章还顺手讲一个每个刷题人迟早会踩的坑：**浮点数算出来的 0.1 + 0.2 不精确等于 0.3**，所以比较浮点结果要用"容差"（允许小误差）而不是直接 `==`。

两道题合起来看是一个重要的选型练习：什么时候用 int、什么时候用 float？什么时候能用 `==` 直接比、什么时候必须带容差？前者是"根据题意选择类型"，后者是"理解每种类型的运算语义"，这两件事贯穿后面所有章节。

## 2. 基础知识：先读懂这些词

- **int（整数）**：Python 里没有大小上限的整数，像 `5`、`-4`、`10**100` 都行。计数、下标、取模这类操作都用它。
- **float（浮点数）**：带小数点的数，比如 `273.15`。它内部是用二进制近似表示的，就像十进制没法精确写出 1/3 一样，二进制也没法精确表示 0.1 这种数，所以浮点运算会带极微小的误差。
- **bool（布尔值）**：只有 `True` 和 `False` 两个值，是比较运算（如 `n % 2 == 0`）的结果，也是 if 分支的判断依据。
- **None**：表示"什么值都没有"。函数没写 return 时默认返回它。
- **整除 `//`**：算商并丢掉小数部分，但注意 Python 是**向负无穷取整**。比如 `-4 // 3` 等于 `-2`（不是 -1），因为 -2 比 -1 更小、更接近负无穷。
- **取模 `%`**：算除法剩下的余数，满足关系 `a == (a//3)*3 + a%3`。跟整除配套，整除往哪边取整，余数就跟着变：`-4 % 3` 等于 `2`，于是 `-4 = (-2)×3 + 2`。
- **向零截断**：另一种丢小数的方式，直接往 0 的方向砍，比如 `-4/3 = -1.333...` 截断成 `-1`。`int(-4/3)` 就是 -1。很多语言（如 C）默认这样做，和 Python 不同，写题时要分清。
- **三元表达式 `x if 条件 else y`**：一行写完"如果条件成立取 x，否则取 y"，相当于压缩版的 if-else。
- **容差（tolerance）**：比较两个浮点数时允许的误差范围，比如 `math.isclose(a, b, abs_tol=1e-9)` 的意思是"a 和 b 的差小于 0.000000001 就算相等"。
- **仿射变换**：形如 `y = kx + b` 的变换（先乘系数再加常数）。摄氏转华氏 `F = 1.8C + 32` 就是典型的仿射变换。
- **类型转换函数**：`int()` 把浮点或字符串转成整数（`int('7')` 得 7），`float()` 转成浮点，`str()` 转成字符串。解析输入、格式化输出时天天用。
- **`raise ValueError`**：主动抛出异常的语句，意思是"这个输入不对，我不算了，请调用方处理"。它把错误暴露在最早的位置，避免错上加错。
- **math.isclose**：标准库提供的浮点比较函数，参数包括绝对容差 `abs_tol` 和相对容差 `rel_tol`，内部帮你判断"两个数是否足够接近"。

## 3. 思路推导：从小例子开始

**思路 A：最小偶数倍。**

- **Step 1：把条件翻译成人话。** 答案要同时满足"是 n 的倍数"和"是偶数"，并且要尽量小。任何答案一定是 n 的 1 倍、2 倍、3 倍……里的某一个。
- **Step 2：分情况讨论。** 如果 n 本身是偶数，那么 n × 1 = n 就同时满足两个条件，而且你不可能找到比 n 更小的"n 的正倍数"，所以答案就是 n。如果 n 是奇数，那么 n 的任何奇数倍（1 倍、3 倍、5 倍……）都还是奇数（奇数乘奇数等于奇数），永远不满足"是偶数"，所以最小的可行选项是 2 倍，即 2n。
- **Step 3：判断奇偶用取模。** `n % 2 == 0` 说明 n 除以 2 没有余数，即 n 是偶数；否则是奇数。再用三元表达式一行写出答案。
- **Step 4：挡住非法输入。** 题目保证 n 是正整数，但我们在函数入口检查 `n < 1` 就直接报错，防止有人传 0 或负数进来得到无意义的结果。

**手算演示一轮（n=5）**：第一步判断奇偶，`5 % 2` 等于 `1`，`1 == 0` 不成立，所以 5 是奇数；第二步走 `2*n` 分支，`2*5 = 10`；第三步验证：10 除以 5 等于 2（是 5 的倍数），10 除以 2 等于 5（是偶数），比 10 小的 5 的倍数只有 5 本身而它是奇数，所以 10 确实是最小偶数倍。再对 n=6 演一遍：`6 % 2 == 0` 成立，直接返回 `6`，6 既是 6 的倍数又是偶数，且没有更小的正倍数。

**思路 B：温度转换。** 就是照公式抄：开氏 = `celsius + 273.15`，华氏 = `celsius*1.8 + 32`，把两个数放进一个列表返回。手算 `celsius=0`：开氏 = 0 + 273.15 = `273.15`；华氏 = 0 × 1.8 + 32 = `32`。注意返回的是两个浮点数，所以测试时要带着容差比较。再手算一个 `celsius=100`（水的沸点，好记）：开氏 = 100 + 273.15 = `373.15`；华氏 = 100 × 1.8 + 32 = 180 + 32 = `212`——212 华氏度正是标准大气压下水的沸点，两个公式互相印证，说明你没记错。

**浮点误差手算演示（为什么要容差）：** 十进制的 0.1 写成二进制是无限循环小数（0.0001100110011...），计算机只能存有限位，所以存进来的 0.1 其实差了一丁点。两个"差一丁点"的数相加，误差就显形了：`0.1 + 0.2` 在 Python 里打印出来是 `0.30000000000000004`，拿它和 `0.3` 用 `==` 比较会得到 False。差值大约是 5.5×10⁻¹⁷，肉眼可忽略，但 `==` 是精确比较、一丁点都不放过，所以必须改用 `math.isclose` 这类"允许误差"的比较。

**负数整除手算演示（“运行示例”部分 表格的关键行）：** 算 `-4 ÷ 3`。如果按"商 -1 余 -1"（向零截断）理解：(-1)×3 + (-1) = -4 也成立；但 Python 的 `//` 向负无穷取整，取商 `-2`，余数用 `a - (a//b)*b` 算出是 `-4 - (-6) = 2`，于是 `-4 % 3 = 2`（余数符号跟除数一致，为正）。两种规则各自自洽，但**不通用**——这就是 “自动测试”部分 最后一条断言把两种结果都锁死的原因。

**整除与取模的对照表（“运行示例”部分 小实例）**：“运行示例”部分 用 `show_table` 画了一张表，对 a 从 -4 到 4 逐个列出 `a // 3`、`a % 3`、以及验证式 `(a//3)*3 + a%3`。挑几行看：`a=4` 时 `4//3=1`、`4%3=1`，验证 1×3+1=4 ✓；`a=-4` 时 `-4//3=-2`、`-4%3=2`，验证 (-2)×3+2=-4 ✓。你看 -4 那行：向负无穷取整得到 -2 而不是 -1，余数相应变成正的 2。这张表就是"Python 整除规则"的实验证据。建议你把 9 行全部读一遍，重点盯住 a 从 -1 跨到 0、从 2 跨到 3 这几个商发生跳变的位置，你会看到余数始终落在 0、1、2 三个值里打转——"余数的绝对值一定小于除数"这条性质在负数区也没有破。

## 4. 核心代码：smallest_even_multiple

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def smallest_even_multiple(n: int) -> int:
    if n < 1:
        raise ValueError('n must be a positive integer')
    return n if n % 2 == 0 else 2 * n
```

### 逐段读懂代码

**核心实现（“分段实现”部分）对应实现：**

```python
def smallest_even_multiple(n: int) -> int:
    if n < 1:
        raise ValueError('n must be a positive integer')
    return n if n % 2 == 0 else 2 * n

def convert_temperature(celsius: float) -> list[float]:
    return [celsius + 273.15, celsius * 1.8 + 32]
```

**函数一：`smallest_even_multiple`**

- `def smallest_even_multiple(n: int) -> int:` —— 定义函数，接收一个整数 n，约定返回整数。
- `if n < 1:` / `raise ValueError('n must be a positive integer')` —— 这是入口守卫：如果 n 小于 1（比如 0 或 -5），就不往下算了，直接抛出一个带提示信息的错误。"最小正偶数倍"对非正数没有意义，早报错早发现。
- `return n if n % 2 == 0 else 2*n` —— 这是本章最紧凑的一行，是三元表达式：条件是 `n % 2 == 0`（n 是偶数）；条件成立时整个表达式的值取 `n`，不成立时取 `2*n`。它等价于四行的 if-else，但一行搞定。逻辑对应第三节的分情况讨论：偶数返回自身，奇数返回两倍。

**函数二：`convert_temperature`**

- `def convert_temperature(celsius: float) -> list[float]:` —— 接收一个浮点数摄氏温度，返回值类型注解写的是 `list[float]`，即"浮点数组成的列表"。
- `return [celsius + 273.15, celsius*1.8 + 32]` —— 一行算两个公式并用列表字面量 `[x, y]` 打包返回：第一个元素是开氏温度（加 273.15），第二个是华氏温度（乘 1.8 再加 32）。没有中间变量，直接把表达式放进列表里返回，这是很常见的紧凑写法。

**两个常见错写，顺手对照一下**：其一，把 `celsius*1.8` 误写成 `celsius*1.80 + 32` 之外的任何系数（比如记成 ×2），拿 celsius=0 测不出来（0 乘什么都得 0，只剩 +32），必须拿 celsius=100 验证 212 才能抓出来——这解释了为什么测公式要选非零输入；其二，把返回列表误写成返回元组 `(a, b)`，解包虽然照样能用，但题目契约要求列表，`==` 比较 `[273.15, 32.0] == (273.15, 32.0)` 在 Python 里是 False，官方判题会判错。

**回看入口守卫为什么放在最前**：`smallest_even_multiple` 的守卫检查的是 n（题目约束"正整数"），而 `convert_temperature` 没有守卫——因为浮点输入对两个公式永远有意义（没有非法值）。这给我们一个通感：守卫不是仪式，只挡"会让答案失去意义"的输入。

## 5. 分段实现：按顺序运行

**本章接口：** `smallest_even_multiple(n)`、`convert_temperature(celsius)`

### smallest_even_multiple

In [1]:
def smallest_even_multiple(n: int) -> int:
    if n < 1:
        raise ValueError('n must be a positive integer')
    return n if n % 2 == 0 else 2 * n

### convert_temperature

In [2]:
def convert_temperature(celsius: float) -> list[float]:
    return [celsius + 273.15, celsius * 1.8 + 32]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['a', 'a // 3', 'a % 3', '验证 a=q×3+r'], [(a,a//3,a%3,(a//3)*3+a%3) for a in range(-4,5)])

a,a // 3,a % 3,验证 a=q×3+r
-4,-2,2,-4
-3,-1,0,-3
-2,-1,1,-2
-1,-1,2,-1
0,0,0,0
1,0,1,1
2,0,2,2
3,1,0,3
4,1,1,4


## 7. 正确性、适用条件与复杂度

奇数 n 的奇数倍仍是奇数，因此至少乘 2；偶数 n 自身可行。Python 整除向负无穷取整，与向零截断不同。

**代价：** 固定字长下 O(1) 时间与空间。

### 展开理解

**最小偶数倍为什么对：** 我们只需要论证两种情况都取到了最小值。当 n 是偶数时，"n 的正倍数"从小到大排列是 n、2n、3n……最小的那个就是 n 自己，而它恰好是偶数，两个条件同时满足，所以 n 就是最优答案，不可能有更小的。当 n 是奇数时，n 的奇数倍（n、3n、5n……）由"奇数乘奇数还是奇数"保证全是奇数，全都不合格；下一个候选就是 2n，而奇数乘 2 一定是偶数，同时 2n 当然还是 n 的倍数，所以 2n 合格，且在 n 的所有倍数里没有比它更小的合格者。两种情况都覆盖了所有可能，所以答案一定正确。

换个角度再体会一遍这个论证的形状：我们没有"搜索"答案，而是先证明答案只可能是两种形态之一，再各给一个"取到最小"的理由。这种"先缩小候选、再逐一击破"的推理方式，比写出代码本身更值得带走——以后很多 O(1) 结论题都是这个套路。

**温度转换为什么对：** 两个公式就是单位制的定义，代码只是照抄定义，没有自由发挥的空间。唯一要注意的是结果是浮点数，比如某些输入下华氏的数学精确值是 36.6，计算机可能算出 36.60000000000001，这不是代码错，是浮点表示的固有近似，所以测试要配容差。

**守卫为什么出现在最小偶数倍里而不出现在温度转换里：** 题目对 n 的约束是"正整数"，n=0 或负数会让"最小正偶数倍"失去意义，所以入口挡住它们是契约的一部分；温度换算函数在这里实现数值公式；物理温度是否低于绝对零度，是另外的输入约束，不能由公式可计算推出物理上有意义。"守卫跟着题意走，不是为了写而写"——这个分寸感是本章暗线之一。

**复杂度：** 两个函数都只做固定次数的加减乘除（求余、比较、两三条公式），固定字长模型下是 O(1)。Python 整数可任意精度；整数运算的实际代价还取决于位数。拿具体数字感受一下：就算 n 是 10⁹，`n % 2` 和一次乘法也是眨眼间完成，不存在随 n 增长的循环。

**整除规则的提醒：** Python 的 `//` 向负无穷取整，`-4//3` 是 `-2`；而 `int(-4/3)` 是先算浮点除法得到 -1.333 再向零截断成 `-1`。两者在负数上不一致，这是边界 bug 的常见来源。

**拿具体数字感受 O(1)：** 我们拿 n 从小到大排一排就能看出"常数时间"是什么意思：n=1 算 1 次，n=10 算 1 次，n=10⁹ 还是算 1 次。操作次数完全不随 n 变化，这就是 O(1) 的直观含义。对比一下本章如果有人用"从 1 往上试每个偶数"的笨办法找最小偶数倍，那可能是 O(n) 级——n=10⁹ 要十亿次操作，秒级都未必扛得住。选对公式（数学推导）和选对循环（暴力扫描）的差距，在这道简单题上就已经体现出来了。

**一个失效条件（对应 “掌握检查”部分 的自测要求）：** 把输入约束改成"n 可以是 0"会怎样？`smallest_even_multiple(0)` 会命中守卫 `n < 1` 抛 ValueError。实现仍然"正确"，因为守卫诚实地说了"不接这种输入"；但如果你悄悄删掉守卫，0 会走 `% 2 == 0` 分支返回 0，而"最小正偶数倍"没有 0 这个答案，错误就被吞掉了。结论：约束一变，守卫和分支要一起重新对表。

## 8. 单元测试：每个用例在检查什么

```python
import math

assert smallest_even_multiple(5) == 10

assert smallest_even_multiple(6) == 6

kelvin, fahrenheit = convert_temperature(0)

assert math.isclose(kelvin, 273.15, abs_tol=1e-09)

assert math.isclose(fahrenheit, 32, abs_tol=1e-09)

assert -4 // 3 == -2 and int(-4 / 3) == -1

print('N002: 所有本章断言通过')
```

- `assert smallest_even_multiple(5) == 10`：正常值 + 奇数分支。5 是奇数，必须走 `2*n` 分支返回 10。如果你忘了奇偶判断直接返回 n，这条会失败。
- `assert smallest_even_multiple(6) == 6`：正常值 + 偶数分支。6 已经是偶数，答案必须是它自己。如果你无脑返回 `2*n`（得 12），这条会失败。这两条合起来把三元表达式的两个分支各测一遍。
- `kelvin, fahrenheit = convert_temperature(0)`：调用转换函数，把返回列表拆开赋给两个变量（这叫解包）。选 celsius=0 是取了"冰点"这个整数边界，两个公式此时分别退化成 273.15 和 32，手算最不容易错。
- `assert math.isclose(kelvin,273.15,abs_tol=1e-9)` 和 `assert math.isclose(fahrenheit,32,abs_tol=1e-9)`：用 `math.isclose` 配上绝对容差 0.000000001 来比较浮点结果。这演示了本章的核心主张：浮点数不要用 `==` 硬比，要给一点误差余地。
- `assert -4//3 == -2 and int(-4/3) == -1`：知识点断言，专门验证"向负无穷取整"和"向零截断"在负数上的区别——同一个 -4/3，`//` 给 -2，`int()` 给 -1。这条不测我们的函数，测的是你对 Python 规则的理解。
- `import math`：注意测试文件开头先导入了标准库 math，因为浮点比较需要 `math.isclose`。测试代码本身也可以有依赖，这很正常。
- 最后的 print：前面全部通过才会执行到这行，看到它就代表整组测试通过。

**故障推演：哪种写错会挂哪条？**

- 把三元表达式写反（`2*n if n % 2 == 0 else n`）：`smallest_even_multiple(5)` 会返回 5 ≠ 10，第一条挂；`smallest_even_multiple(6)` 返回 12 ≠ 6，第二条也挂——两个分支各有一条断言把守，写反必被抓；
- 华氏公式记错成 `celsius*2 + 32`：拿 celsius=0 测不出（32 碰巧对），这就是这条测试的盲区；本次已增加一条 `convert_temperature(100)` 的检查（期望 212），用于补齐原测试的盲区；
- 忘了 Python 负数取模规则、以为 `-4//3 == -1`：最后一条知识点断言直接挂，它不测你的函数，专测你对语言规则的记忆。

**这组断言的分层小结**：两条函数断言管"正常值 + 分支覆盖"，两条 isclose 断言管"浮点语义"，最后一条管"语言规则"。三个层次各司其职，这就是一套微型但完整的测试观。

**断言分层一览表**：

| 断言 | 层次 | 它守护的契约 |
| --- | --- | --- |
| `smallest_even_multiple(5) == 10` | 正常值·奇数分支 | 奇数必须翻倍 |
| `smallest_even_multiple(6) == 6` | 正常值·偶数分支 | 偶数必须原样返回 |
| `isclose(kelvin,273.15)` / `isclose(fahrenheit,32)` | 浮点语义 | 公式正确 + 用容差比较 |
| `-4//3 == -2 and int(-4/3) == -1` | 语言规则 | Python 负数取整的知识点 |

In [4]:
import math

assert smallest_even_multiple(5) == 10

assert smallest_even_multiple(6) == 6

kelvin, fahrenheit = convert_temperature(0)

assert math.isclose(kelvin, 273.15, abs_tol=1e-09)

assert math.isclose(fahrenheit, 32, abs_tol=1e-09)

assert -4 // 3 == -2 and int(-4 / 3) == -1

# 冰点不足以区分华氏公式的斜率，增加非零摄氏度用例。
assert math.isclose(convert_temperature(100)[1], 212, abs_tol=1e-9)
try:
    smallest_even_multiple(0)
except ValueError:
    pass
else:
    raise AssertionError('必须拒绝不符合正整数契约的 0')
assert (2 * 5 if 5 % 2 == 0 else 5) == 5

print('N002: 所有本章断言通过')

N002: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 解释负数整除与向零截断的差异。

**练习 2：** 比较浮点结果时选择容差。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（解释负数整除与向零截断的差异）**：提示——围绕"商往哪边靠"来讲。向零截断是把 -1.333 砍成 -1（往 0 靠），向负无穷取整是取 -2（往更小的方向靠）。你可以拿 “运行示例”部分 表格里 `a=-4` 和 `a=-1` 两行当手算示例：`-4//3=-2, -4%3=2`，验证 (-2)×3+2=-4；而 `int(-4/3)=-1`。写验证代码时可以直接断言这两个结果，再总结一句"余数符号跟随除数、由取整方向决定"。验收时给自己加三个小问：`7 // 2` 是几（正数无争议，答案是 3）；`-7 // 2` 是几（-4，不是 -3）；`-7 % 2` 是几（1，因为 -7 = (-4)×2 + 1）。三问答对，说明规则真的进脑子了。
- **练习 2（比较浮点结果时选择容差）**：提示——先做一个反例：断言 `0.1 + 0.2 == 0.3` 会失败，打印 `0.1 + 0.2 - 0.3` 你会看到 5.5e-17 这种极小残差。然后改用 `math.isclose`，并思考容差取多大合适：本题温度值在几十到几百的量级，`abs_tol=1e-9` 够用；如果是天文数字或接近 0 的值，可能要考虑同时给相对容差 `rel_tol`。边界示例可以用 celsius=0（期望 273.15 和 32）和 celsius=-273.15（期望开氏为 0）。验收时解释清楚一件事就够：容差不是拍脑袋——它应当显著大于"计算可能积累的误差"，又显著小于"你关心的真实差异"，本题 1e-9 正好卡在这个带子里。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def smallest_even_multiple(n: int) -> int:
    if n < 1:
        raise ValueError('n must be a positive integer')
    return n if n % 2 == 0 else 2 * n

def convert_temperature(celsius: float) -> list[float]:
    return [celsius + 273.15, celsius * 1.8 + 32]

# 示例与回归测试
import math

assert smallest_even_multiple(5) == 10

assert smallest_even_multiple(6) == 6

kelvin, fahrenheit = convert_temperature(0)

assert math.isclose(kelvin, 273.15, abs_tol=1e-09)

assert math.isclose(fahrenheit, 32, abs_tol=1e-09)

assert -4 // 3 == -2 and int(-4 / 3) == -1

# 冰点不足以区分华氏公式的斜率，增加非零摄氏度用例。
assert math.isclose(convert_temperature(100)[1], 212, abs_tol=1e-9)
try:
    smallest_even_multiple(0)
except ValueError:
    pass
else:
    raise AssertionError('必须拒绝不符合正整数契约的 0')
assert (2 * 5 if 5 % 2 == 0 else 5) == 5

print('N002: 所有本章断言通过')

N002: 所有本章断言通过


## 11. 代表题与迁移

- **2413. Smallest Even Multiple**：给正整数 n 求最小偶数倍。这题练的就是本章的奇偶判断加两种情况的推理（偶数返回 n、奇数返回 2n），对应 `smallest_even_multiple` 的完整逻辑。很多同学第一次会把题目想复杂成"循环枚举 n 的倍数再挑偶数"，写完对比本章的一行三元表达式，正好体会"先推理再编码"能省多少事。
- **2469. Convert the Temperature**：给摄氏度，返回 [开氏, 华氏]。这题练的是"照公式写数值表达式 + 用列表打包多返回值"，对应 `convert_temperature`；提交后你会看到官方判题器对浮点比较本身就带容差，这正好呼应本章的浮点近似知识点。

**提交前的自检三问**：第一，偶数倍那题我有没有把 15 这种"同时满足多条件"的思路迁移过来（本章是 2 的倍数，逻辑同类）？第二，温度那题返回的是列表而不是元组吗？第三，浮点比较我有没有顺手用 `math.isclose` 代替 `==` 的意识？这三问分别对应本章的类型选择、契约意识和浮点语义三个知识点。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。